# Crawling data

Dilakukan crawling data sebanyak 200 data dari detik.com dengan ketentuan 100 data dengan kategori sport dan 100 data dengan kategori finance

In [13]:
%pip install pandas

^C
Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import csv
import time
import pandas as pd

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ----

Dibawah merupakan kode untuk melakukan Pengumpulan URL (Crawling)

In [15]:
def get_detik_urls(base_index_url, target_count=100):

    urls = []
    page = 1
    
    print(f"Mengumpulkan URL dari {base_index_url}...")
    while len(urls) < target_count:
        # Format pagination URL indeks detik
        url = f"{base_index_url}?page={page}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Detik biasanya menyimpan judul dan link berita pada elemen <a> di dalam <article> atau class media__title
        articles = soup.find_all('article')
        
        for article in articles:
            link_tag = article.find('a')
            if link_tag and link_tag.has_attr('href'):
                link = link_tag['href']
                
                # Filter URL: pastikan itu dari detik.com dan hindari format foto/video
                if 'detik.com' in link and 'foto' not in link and 'video' not in link:
                    if link not in urls:
                        urls.append(link)
                        
                if len(urls) >= target_count:
                    break
                    
        page += 1
        time.sleep(1) # Delay agar tidak membebani server (polite crawling)
        
    return urls

Dibawah ini merupakan kode untuk melakukan Ekstraksi Teks (Scraping)

In [16]:
def scrape_articles_content(urls, category):
    """
    Mengekstrak teks konten dari setiap URL menggunakan Trafilatura.
    """
    data = []
    print(f"\n🚀 Memulai ekstraksi {len(urls)} artikel [{category.upper()}]...")
    
    for idx, url in enumerate(urls, 1):
        # Indikator progres yang rapi di satu baris
        print(f"   └─ [{idx:03d}/{len(urls):03d}] Processing: {url[:50]}...", end="\r")
        
        try:
            # Download HTML halaman
            downloaded = trafilatura.fetch_url(url)
            
            if downloaded:
                # Ekstrak teks utama (mengabaikan komentar & tabel)
                text = trafilatura.extract(
                    downloaded, 
                    include_comments=False, 
                    include_tables=False,
                    no_fallback=False
                )
                
                if text:
                    data.append({
                        'isi_berita': text,
                        'label': category,
                        'url': url  # Disimpan sebagai referensi
                    })
        except Exception as e:
            # Lewati URL jika terjadi kegagalan koneksi
            continue
            
        time.sleep(0.3)
        
    print(f"\n✅ Selesai! Berhasil mengekstrak {len(data)} konten [{category.upper()}].")
    return data

In [19]:
# 1. Ambil URL
url_sport = get_detik_urls('https://sport.detik.com/indeks', 100)
print(f"-> Total URL Sport yang didapatkan: {len(url_sport)}")

# Tampilkan sampel 5 URL Sport pertama
print("\n--- Sampel URL Sport (5 Pertama) ---")
for i, url in enumerate(url_sport[:5], 1):
    print(f"{i}. {url}")

print("\n" + "="*50 + "\n")

url_finance = get_detik_urls('https://finance.detik.com/indeks', 100)
print(f"-> Total URL Finance yang didapatkan: {len(url_finance)}")

# Tampilkan sampel 5 URL Finance pertama
print("\n--- Sampel URL Finance (5 Pertama) ---")
for i, url in enumerate(url_finance[:5], 1):
    print(f"{i}. {url}")

Mengumpulkan URL dari https://sport.detik.com/indeks...
-> Total URL Sport yang didapatkan: 100

--- Sampel URL Sport (5 Pertama) ---
1. https://sport.detik.com/sport-lain/d-8662678/tak-takut-tekanan-jepang-forki-targetkan-1-emas-di-asian-games-2026
2. https://sport.detik.com/moto-gp/d-8662441/crash-di-san-marino-bezzecchi-balapan-masih-banyak
3. https://sport.detik.com/moto-gp/d-8662260/karena-jorge-martin-kehilangan-kendali
4. https://sport.detik.com/moto-gp/d-8661830/bos-ducati-marc-marquez-bukan-favorit-juara
5. https://sport.detik.com/moto-gp/d-8661491/pimpin-klasemen-marc-marquez-kejuaraan-belum-berakhir


Mengumpulkan URL dari https://finance.detik.com/indeks...
-> Total URL Finance yang didapatkan: 100

--- Sampel URL Finance (5 Pertama) ---
1. https://finance.detik.com/berita-ekonomi-bisnis/d-8662707/menkeu-suahasil-defisit-apbn-akan-tetap-di-bawah-3
2. https://finance.detik.com/moneter/d-8662772/penyaluran-kur-tembus-rp-219-62-triliun-diterima-3-41-juta-debitur
3. https://fin

In [22]:
import pandas as pd

# 1. Ekstrak Konten Berita
print("🚀 Memulai ekstraksi konten...")
data_sport = scrape_articles_content(url_sport, 'sport')
data_finance = scrape_articles_content(url_finance, 'finance')

# 2. Cek dan Hapus Duplikat
df_raw = pd.DataFrame(data_sport + data_finance)

# Hitung jumlah duplikat berdasarkan URL dan Isi Berita
dup_url = df_raw.duplicated(subset=['url']).sum()
dup_isi = df_raw.duplicated(subset=['isi_berita']).sum()

# Hapus data duplikat (menyimpan entri pertama)
df_clean = df_raw.drop_duplicates(subset=['isi_berita'], keep='first').reset_index(drop=True)

# Ringkasan Hasil
print("\n--- LAPORAN EKSTRAKSI & PENGECEKAN DUPLIKAT ---")
print(f"• Total berita diekstrak  : {len(df_raw)}")
print(f"• Duplikat berdasarkan URL : {dup_url}")
print(f"• Duplikat isi berita      : {dup_isi}")
print(f"• Total data bersih (unik) : {len(df_clean)}")

🚀 Memulai ekstraksi konten...

🚀 Memulai ekstraksi 100 artikel [SPORT]...
   └─ [100/100] Processing: https://sport.detik.com/f1/d-8651642/mamma-mia-ant...
✅ Selesai! Berhasil mengekstrak 100 konten [SPORT].

🚀 Memulai ekstraksi 100 artikel [FINANCE]...
   └─ [100/100] Processing: https://finance.detik.com/berita-ekonomi-bisnis/d-...
✅ Selesai! Berhasil mengekstrak 100 konten [FINANCE].

--- LAPORAN EKSTRAKSI & PENGECEKAN DUPLIKAT ---
• Total berita diekstrak  : 200
• Duplikat berdasarkan URL : 0
• Duplikat isi berita      : 0
• Total data bersih (unik) : 200


In [25]:
# 3. Gabungkan Data & Simpan ke CSV dengan Pandas
all_data = data_sport + data_finance
df = pd.DataFrame(all_data)

# Simpan CSV dengan encoding utf-8-sig
df.to_csv('detik_sport_finance_200.csv', index=False, encoding='utf-8-sig')

print("\nProses Selesai!")
print(f"Total baris data tersimpan: {len(df)}")


Proses Selesai!
Total baris data tersimpan: 200
